# COLMAP dense MVS + Gaussian Splatting on a Colab GPU

**Status: written, not yet run.** Everything in the repository's README was produced on an Apple M5
without CUDA. COLMAP's PatchMatch stereo and Gaussian Splatting need an NVIDIA GPU, so this
notebook is the stretch goal for a free Colab T4. Numbers from it are *not* in the README until
you run it.

What it does, for one ETH3D scene:
1. downloads the scene (same files as `mv3d-download eth3d`);
2. runs COLMAP sparse SfM with the known intrinsics, then PatchMatch stereo + fusion (dense);
3. scores the fused cloud against the laser scan with the repo's own `cloud_metrics`, so it is
   directly comparable with the plane-sweep results in `reports/dense/results.json`;
4. trains a Gaussian Splatting model (nerfstudio `splatfacto`) on the same images and reports
   held-out PSNR (novel-view quality; a different question than geometric accuracy).

In [ ]:
!nvidia-smi
!apt-get -qq install -y colmap p7zip-full > /dev/null
!git clone -q https://github.com/haroonkhan02/multiview-3d-reconstruction-blender.git
%cd multiview-3d-reconstruction-blender
!pip -q install -e ".[full]"

In [ ]:
SCENE = "relief"   # pipes | meadow | terrace | relief
!mv3d-download eth3d --scenes $SCENE

## 1 · COLMAP sparse + dense (CUDA PatchMatch)

In [ ]:
import numpy as np, subprocess, pathlib
from mvrecon.datasets import load_scene
from mvrecon.colmap_sfm import write_resized
scene = load_scene(f"eth3d/{SCENE}", max_side=1600)
work = pathlib.Path(f"outputs/colab/{SCENE}"); img = write_resized(scene, work / "images")
K = scene.K[0]; params = f"{K[0,0]},{K[1,1]},{K[0,2]},{K[1,2]}"
def run(cmd): print(cmd); subprocess.run(cmd, shell=True, check=True)
run(f"colmap feature_extractor --database_path {work}/db.db --image_path {img} "
    f"--ImageReader.camera_model PINHOLE --ImageReader.single_camera 1 --ImageReader.camera_params {params}")
run(f"colmap exhaustive_matcher --database_path {work}/db.db")
(work / "sparse").mkdir(exist_ok=True)
run(f"colmap mapper --database_path {work}/db.db --image_path {img} --output_path {work}/sparse "
    "--Mapper.ba_refine_focal_length 0 --Mapper.ba_refine_principal_point 0 --Mapper.ba_refine_extra_params 0")
run(f"colmap image_undistorter --image_path {img} --input_path {work}/sparse/0 --output_path {work}/dense")
run(f"colmap patch_match_stereo --workspace_path {work}/dense")
run(f"colmap stereo_fusion --workspace_path {work}/dense --output_path {work}/dense/fused.ply")

## 2 · Score against the laser scan
The COLMAP model lives in its own frame: align its camera centres to the ground truth (Umeyama),
move the fused cloud with the same similarity, then use the repo's metrics.

In [ ]:
import open3d as o3d, pycolmap
from mvrecon.colmap_sfm import to_reconstruction
from mvrecon.evaluate import align_and_compare, apply_sim3, cloud_metrics, load_scan
rec = to_reconstruction(pycolmap.Reconstruction(str(work / "sparse/0")), scene, {})
m = align_and_compare(rec.R, rec.t, scene.R_gt, scene.t_gt)
pts = apply_sim3(np.asarray(o3d.io.read_point_cloud(str(work / "dense/fused.ply")).points), m["sim3"])
res = cloud_metrics(pts, load_scan(scene))
print({k: v for k, v in res.items() if k.startswith("tau")})

## 3 · Gaussian Splatting (nerfstudio splatfacto)

In [ ]:
!pip -q install nerfstudio
!ns-process-data images --data {work}/images --output-dir {work}/ns --skip-colmap --colmap-model-path ../sparse/0 || true
!ns-train splatfacto --data {work}/ns --max-num-iterations 15000 --viewer.quit-on-train-completion True
!ns-eval --load-config $(ls -d outputs/ns/splatfacto/*/config.yml | tail -1) --output-path {work}/splat_eval.json
print(open(f"{work}/splat_eval.json").read())

Record the numbers (accuracy / completeness / F1 at 2, 5, 10 cm for COLMAP dense; PSNR / SSIM /
LPIPS for splatfacto) together with the GPU type and run time before adding them to the README.